In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np


# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)





df_raw = df_raw.drop(columns=["Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
source_cities = ['Houston', 'Philadelphia', 'Phoenix', 'San Jose', 'Jacksonville', 'Austin']
target_cities = ['New York', 'Los Angeles', 'Chicago']


source_df = df_raw[df_raw['city'].isin(source_cities)].copy()
target_df = df_raw[df_raw['city'].isin(target_cities)].copy()

# 2. ONE-HOT encoding
cols_to_encode = ["city", "Type", "weather_condition", "Store", "Dept"]

source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)
TARGET = "Weekly_Sales"
# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")


lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}
model = LGBMRegressor(**lgb_params)
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)


# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test_combined)

mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
Source train samples: 184166
Source test samples : 20463
Target train samples: 21409
Target test samples : 192689

Final train set: 205575
Final test set : 213152
 Original Source samples (100%): 204629
 Target samples added to train set (10%): 21409
 Remaining Target samples for testing (90%): 192689
MAE : 3714.91
RMSE: 6326.85
R²  : 0.9399


In [9]:

from sklearn.metrics import mean_absolute_error


import pandas as pd
import numpy as np

# Path to your specific Walmart CSV file
CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(
    columns=[ "Date", "Season", "DayOfWeek", "Month",
             "WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Define domains based on city
source_weather = ['Clouds', 'Rain', 'Snow']
target_weather = ['Clear']


source_df = df_raw[df_raw['weather_condition'].isin(source_weather)].copy()
target_df = df_raw[df_raw['weather_condition'].isin(target_weather)].copy()

# 2. ONE-HOT encoding
cols_to_encode = ["city", "Type", "weather_condition", "Store", "Dept"]

source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)

TARGET = "Weekly_Sales"

# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")
lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)

print(" Training LightGBM on ALL source + 10% target domain...")
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)


# Predict ONLY on the 90% test portion to avoid data leakage
y_pred = model.predict(X_test_combined)

mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
Source train samples: 200081
Source test samples : 22232
Target train samples: 2847
Target test samples : 25629

Final train set: 202928
Final test set : 47861
 Original Source samples (100%): 222313
 Target samples added to train set (10%): 2847
 Remaining Target samples for testing (90%): 25629
 Training LightGBM on ALL source + 10% target domain...
MAE : 2288.15
RMSE: 4204.64
R²  : 0.9668


In [10]:
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np



CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()
means = df_raw.groupby("Is_Christmas_Season")["Weekly_Sales"].mean()
std = df_raw.groupby("Is_Christmas_Season")["Weekly_Sales"].std()
print(" Weekly_Sales Mean by Christmas Season:\n", means)
print(" Weekly_Sales Std by Christmas Season:\n", std)

cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
df_encoded = pd.get_dummies(df_raw, columns=cols_to_encode)
df_encoded = df_encoded.fillna(0)

# Source Domain: Normal Days (In-Distribution)
# Target Domain: Holidays (Out-of-Distribution)
source_df = df_encoded[df_encoded['Is_Christmas_Season'] == 0].copy()
target_df = df_encoded[df_encoded['Is_Christmas_Season'] == 1].copy()

# Define feature columns (exclude target, proxies, and the splitting variable)
TARGET = "Weekly_Sales"
# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")
from lightgbm import LGBMRegressor

lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)
print(" Training LightGBM on source domain...")
model.fit(X_train_combined , y_train_combined)
y_pred = model.predict(X_test_combined)
from sklearn.metrics import mean_squared_error, r2_score

mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")



 Loading Walmart Dataset from CSV...
 Weekly_Sales Mean by Christmas Season:
 Is_Christmas_Season
0.0    15825.036257
1.0    20540.453185
Name: Weekly_Sales, dtype: float64
 Weekly_Sales Std by Christmas Season:
 Is_Christmas_Season
0.0    22336.091663
1.0    29823.787737
Name: Weekly_Sales, dtype: float64
Source train samples: 360803
Source test samples : 40090
Target train samples: 1783
Target test samples : 16051

Final train set: 362586
Final test set : 56141
 Original Source samples (100%): 400893
 Target samples added to train set (10%): 1783
 Remaining Target samples for testing (90%): 16051
 Training LightGBM on source domain...
MAE : 3231.22
RMSE: 7243.30
R²  : 0.9157


In [11]:
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np




CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()
df_raw['Store'] = df_raw['Store'].astype(int)



source_df = df_raw[df_raw['Store'].between(1, 30)].copy()
target_df = df_raw[df_raw['Store'].between(31, 45)].copy()
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)
# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")
from lightgbm import LGBMRegressor

lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)
print(" Training LightGBM on source domain...")
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)
y_pred = model.predict(X_test_combined)
from sklearn.metrics import mean_squared_error, r2_score

mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

 Loading Walmart Dataset from CSV...
Source train samples: 262671
Source test samples : 29186
Target train samples: 12687
Target test samples : 114183

Final train set: 275358
Final test set : 143369
 Original Source samples (100%): 291857
 Target samples added to train set (10%): 12687
 Remaining Target samples for testing (90%): 114183
 Training LightGBM on source domain...
MAE : 3406.79
RMSE: 5844.59
R²  : 0.9183


In [12]:
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np



CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.drop(columns=[ "Date","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

seasons_124 = df_raw[df_raw['Season'].isin([1,2,4])]['Weekly_Sales']
mean_124 = seasons_124.mean()
std_124 = seasons_124.std()

seasons_3 = df_raw[df_raw['Season'] == 3]['Weekly_Sales']
mean_3 = seasons_3.mean()
std_3 = seasons_3.std()

print("Season 1,2,4 -> mean:", mean_124, ", std:", std_124)
print("Season 3 -> mean:", mean_3, ", std:", std_3)

# Manual One-Hot Encoding


# Source Domain: Seasons 1, 2, 4 (In-Distribution)
# Target Domain: Season 3 (Out-of-Distribution)
source_df = df_raw[df_raw['Season'].isin([1, 2,4])].copy()
target_df = df_raw[df_raw['Season'].isin([3])].copy()
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)
# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")
from lightgbm import LGBMRegressor

lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)
print(" Training LightGBM on source domain...")
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)
y_pred = model.predict(X_test_combined)
from sklearn.metrics import mean_squared_error, r2_score

mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")



 Loading Walmart Dataset from CSV...
Season 1,2,4 -> mean: 16142.152215400898 , std: 23077.433816158824
Season 3 -> mean: 15723.876093791609 , std: 21781.446728632603
Source train samples: 272088
Source test samples : 30232
Target train samples: 11640
Target test samples : 104767

Final train set: 283728
Final test set : 134999
 Original Source samples (100%): 302320
 Target samples added to train set (10%): 11640
 Remaining Target samples for testing (90%): 104767
 Training LightGBM on source domain...
MAE : 2449.58
RMSE: 4068.11
R²  : 0.9656


In [13]:
import pandas as pd





CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"


print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

# Clean up columns
df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()

# Manual One-Hot Encoding


# Source Domain: Store Type C (In-Distribution)
# Target Domain: Store Types A and B (Out-of-Distribution)
source_df = df_raw[df_raw['Type'].isin(['C'])].copy()
target_df = df_raw[df_raw['Type'].isin(['A','B'])].copy()
cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
source_df = pd.get_dummies(source_df, columns=cols_to_encode)
target_df = pd.get_dummies(target_df, columns=cols_to_encode)
source_df, target_df = source_df.align(target_df, join='left', axis=1, fill_value=0)
# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")
from lightgbm import LGBMRegressor

lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)
print(" Training LightGBM on source domain...")
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)
y_pred = model.predict(X_test_combined)
from sklearn.metrics import mean_squared_error, r2_score


mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")



 Loading Walmart Dataset from CSV...
Source train samples: 38225
Source test samples : 4248
Target train samples: 37625
Target test samples : 338629

Final train set: 75850
Final test set : 342877
 Original Source samples (100%): 42473
 Target samples added to train set (10%): 37625
 Remaining Target samples for testing (90%): 338629
 Training LightGBM on source domain...
MAE : 3404.08
RMSE: 6307.25
R²  : 0.9261


In [14]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np



CSV_PATH = r"D:\walmart_dataset\final_data_walmart.csv"

print(" Loading Walmart Dataset from CSV...")
df_raw = pd.read_csv(CSV_PATH)

df_raw = df_raw.drop(columns=[ "Date","Season","DayOfWeek","Month","WeekOfYear"], errors='ignore')
df_raw.columns = df_raw.columns.str.strip()
means = df_raw.groupby("IsHoliday")["Weekly_Sales"].mean()
std = df_raw.groupby("IsHoliday")["Weekly_Sales"].std()
print(" Weekly_Sales Mean by IsHoliday:\n", means)
print(" Weekly_Sales Std by IsHoliday:\n", std)

cols_to_encode = ["city", "Type", "weather_condition",
                    "Store", "Dept"]
df_encoded = pd.get_dummies(df_raw, columns=cols_to_encode)
df_encoded = df_encoded.fillna(0)

# Source Domain: Normal Days (In-Distribution)
# Target Domain: Holidays (Out-of-Distribution)
TARGET = "Weekly_Sales"

source_df = df_encoded[df_encoded['IsHoliday'] == 0].copy().reset_index(drop=True)
target_df = df_encoded[df_encoded['IsHoliday'] == 1].copy().reset_index(drop=True)

# Features / target
X_source = source_df.drop(columns=[TARGET])
y_source = source_df[TARGET]

X_target = target_df.drop(columns=[TARGET])
y_target = target_df[TARGET]

# Clean column names




# SOURCE DOMAIN: 90 train / 10 test

X_source_train, X_source_test, y_source_train, y_source_test = train_test_split(
    X_source,
    y_source,
    test_size=0.1,
    random_state=42
)


# TARGET DOMAIN: 10 train / 90 test

X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_target,
    y_target,
    train_size=0.1,
    random_state=42
)


# FINAL TRAIN SET
# 90% source + 10% target

X_train_combined = pd.concat(
    [X_source_train, X_target_train],
    ignore_index=True
)

y_train_combined = pd.concat(
    [y_source_train, y_target_train],
    ignore_index=True
)


# FINAL TEST SET
# 10% source + 90% target

X_test_combined = pd.concat(
    [X_source_test, X_target_test],
    ignore_index=True
)

y_test_combined = pd.concat(
    [y_source_test, y_target_test],
    ignore_index=True
)

# Shuffle train/test
train_idx = np.random.RandomState(42).permutation(len(X_train_combined))
X_train_combined = X_train_combined.iloc[train_idx].reset_index(drop=True)
y_train_combined = y_train_combined.iloc[train_idx].reset_index(drop=True)

test_idx = np.random.RandomState(42).permutation(len(X_test_combined))
X_test_combined = X_test_combined.iloc[test_idx].reset_index(drop=True)
y_test_combined = y_test_combined.iloc[test_idx].reset_index(drop=True)

# Info
print(f"Source train samples: {len(X_source_train)}")
print(f"Source test samples : {len(X_source_test)}")

print(f"Target train samples: {len(X_target_train)}")
print(f"Target test samples : {len(X_target_test)}")

print(f"\nFinal train set: {len(X_train_combined)}")
print(f"Final test set : {len(X_test_combined)}")

print(f" Original Source samples (100%): {len(X_source)}")
print(f" Target samples added to train set (10%): {len(X_target_train)}")
print(f" Remaining Target samples for testing (90%): {len(X_target_test)}")

# Combine ALL source data with the 10% portion of target data

from lightgbm import LGBMRegressor

lgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'num_leaves': 62,
    'max_depth': 20,
    'verbose': -1,
    'n_jobs': 1
}

model = LGBMRegressor(**lgb_params)
X_train_combined = X_train_combined.apply(pd.to_numeric, errors='coerce')
X_test_combined = X_test_combined.apply(pd.to_numeric, errors='coerce')

X_train_combined = X_train_combined.fillna(0).astype(np.float32)
X_test_combined = X_test_combined.fillna(0).astype(np.float32)
model.fit(X_train_combined, y_train_combined)
y_pred = model.predict(X_test_combined)
from sklearn.metrics import mean_squared_error, r2_score


mae = mean_absolute_error(y_test_combined, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_combined, y_pred))
r2 = r2_score(y_test_combined, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")


 Loading Walmart Dataset from CSV...
 Weekly_Sales Mean by IsHoliday:
 IsHoliday
False    15944.465963
True     17108.099010
Name: Weekly_Sales, dtype: float64
 Weekly_Sales Std by IsHoliday:
 IsHoliday
False    22343.199043
True     27279.158431
Name: Weekly_Sales, dtype: float64
Source train samples: 350490
Source test samples : 38944
Target train samples: 2929
Target test samples : 26364

Final train set: 353419
Final test set : 65308
 Original Source samples (100%): 389434
 Target samples added to train set (10%): 2929
 Remaining Target samples for testing (90%): 26364
MAE : 3121.19
RMSE: 8787.49
R²  : 0.8717
